In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import dask.dataframe as dd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
import random
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV


In [ ]:
df = pd.read_csv('/home/harsh/Desktop/COEN 281 Pattern Reco and Data mining/Homework/Project/sampled_data2.csv')

In [ ]:
df['hour'] = df['hour'].astype(str).str.zfill(8)
df['year'] = '20'+df['hour'].str[:2]
df['month'] = df['hour'].str[2:4]
df['day'] = df['hour'].str[4:6]
df['hour_of_day'] = df['hour'].str[6:8]
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)
df['day'] = df['day'].astype(int)
df['hour_of_day'] = df['hour_of_day'].astype(int)


In [ ]:
# click label imbalance
import seaborn as sns
import matplotlib as plt

sns.countplot(x='click', data=df)
plt.show()

In [ ]:

if 'id' in df:
    df = df.drop(columns=['id'])


In [ ]:
df = pd.get_dummies(df, columns=['site_category', 'app_category'], drop_first=True)

In [ ]:
df['site_domain_freq'] = df['site_domain'].map(df['site_domain'].value_counts())
df['app_domain_freq'] = df['app_domain'].map(df['app_domain'].value_counts())
df['device_ip_freq'] = df['device_ip'].map(df['device_ip'].value_counts())
df['site_domain_freq'] = df['site_domain'].map(df['site_domain'].value_counts())
df['app_domain_freq'] = df['app_domain'].map(df['app_domain'].value_counts())
df['device_ip_freq'] = df['device_ip'].map(df['device_ip'].value_counts())

In [ ]:
if 'device_id' in df:
    df = df.drop(columns=['device_id'])
if 'device_ip' in df:
    df = df.drop(columns=['device_ip'])
if 'site_domain' in df:
    df = df.drop(columns=['site_domain'])
if 'app_id' in df:
    df = df.drop(columns=['app_id'])
if 'device_model' in df:
    df = df.drop(columns=['device_model'])
if 'C14' in df:
    df = df.drop(columns=['C14'])
if 'site_id' in df:
    df = df.drop(columns=['site_id'])
if 'hour' in df:
    df = df.drop(columns=['hour'])
if 'app_domain' in df:
    df = df.drop(columns=['app_domain'])

In [ ]:
unique_dates = df['day'].unique()

split_index = int(0.8 * len(unique_dates))

train_dates = unique_dates[:split_index]
test_dates = unique_dates[split_index:]

train_df = df[df['day'].isin(train_dates)]
test = df[df['day'].isin(test_dates)]

In [ ]:
train_df.head()

In [ ]:
x_train = train_df.drop('click', axis=1)  
y_train = train_df['click']

In [ ]:
rf_grid = RandomForestClassifier(max_depth= 7 ,n_estimators=40, class_weight={0: 1, 1: 2},min_samples_leaf=1)
model_grid = rf_grid.fit(x_train, y_train)


In [ ]:
gbdt_grid = XGBClassifier(eval_metric='logloss', random_state=42,colsample_bytree=0.6, learning_rate=0.2, max_depth=5, min_child_weight=5, n_estimators=50, subsample=0.8)
model_grid_gbdt = gbdt_grid.fit(x_train, y_train)

In [ ]:
X_test = test.drop(columns=['click'])
y_test = test['click']

In [ ]:
y_pred_proba = model_grid_gbdt.predict_proba(X_test)[:, 1]  
y_pred = model_grid_gbdt.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")

In [ ]:
y_pred_proba = model_grid.predict_proba(X_test)[:, 1]  
y_pred = model_grid.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")